# Notebook 02 — Ingest Patents from Lens.org

This notebook downloads synthetic biology patents from the Lens.org API
and saves a normalized CSV.

**Before running:**
- Set `LENS_API_TOKEN` in your `.env` file
- Get a free token at https://www.lens.org/lens/user/subscriptions

In [ ]:
import sys
sys.path.insert(0, '..')

from pathlib import Path
from src.utils.config import load_config
from src.ingest import lens, normalize

cfg = load_config()
print('Config loaded.')

In [ ]:
# --- 1. Fetch patents from Lens.org ---

raw_patents = lens.search_patents(
    keywords=cfg['corpus']['seed_keywords'],
    year_min=cfg['corpus']['year_min'],
    year_max=cfg['corpus']['year_max'],
    max_results=cfg['corpus']['lens_max_results'],
)

raw_records = [lens.extract_fields(p) for p in raw_patents]
print(f'Downloaded {len(raw_records)} patents from Lens.org.')

In [ ]:
# --- 2. Normalize and tag ---

patents_df = normalize.normalize_patents(
    raw_records=raw_records,
    carbon_keywords=cfg['corpus']['carbon_capture_keywords'],
)

print(f'Normalized: {len(patents_df)} patents')
print(f'Carbon capture: {patents_df["case_study_flag"].sum()} patents tagged')
patents_df.head()

In [ ]:
output_path = Path('../data/processed/patents.csv')
output_path.parent.mkdir(parents=True, exist_ok=True)
patents_df.to_csv(output_path, index=False)
print(f'Saved to {output_path}')